## Neural Network with Pytorch

### Roadmap

#### Phase 1 - Data preparation for PyTorch
* Convert DataFrames to tensors
* Create Dataset & DataLoader
* Handle categorical encoding already done

#### Phase 2 - Neural network architecture
* Simple but effective Multilayer Perceptron (MLP)
* Proper activation functions
* Regularization

#### Phase 3 - Training loop
* Loss function
* Optimizer
* Validation monitoring
* Early stopping

#### Phase 4 - Evaluation
* Accuracy
* ROC-AUC
* Comparison vs Logistic Regression

#### Phase 5 - Interpretation
* Why NN helps / doesn’t help
* When NN is worth it in sports data

----------------------------------------------------------------------------------------------------------

### Phase 1 - Data Preparation

In [21]:
import sys
from pathlib import Path

# Regarding Tennis_ML_Project/Notebooks, scr is a sibbling, not a child, 
# so we need to add the parent of Notebooks to the path so that we can import from src

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))


In [22]:
import src
print("src imported correctly:", src)


src imported correctly: <module 'src' from 'c:\\Users\\X421IA\\Desktop\\Learning Projects\\Tennis_ML_Project\\src\\__init__.py'>


In [23]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Import our own modules
from src.dataset import MatchDataset
from src.models import MatchOutcomeNN
from src.train import train_one_epoch, evaluate


In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


#### Load processed data

In [25]:
X_train_df = pd.read_csv("../data/processed/X_train.csv")
X_val_df = pd.read_csv("../data/processed/X_val.csv")
X_test_df = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_val = pd.read_csv("../data/processed/y_val.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print(X_train_df.shape, X_val_df.shape, X_test_df.shape)


(25630, 12) (5972, 12) (6152, 12)


#### Convert to PyTorch tensors

In [26]:
X_train_t = torch.tensor(X_train_df.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

X_val_t = torch.tensor(X_val_df.values, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

X_test_t = torch.tensor(X_test_df.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

print(X_train_t.shape, y_train_t.shape)


torch.Size([25630, 12]) torch.Size([25630])


#### Create datasets and dataloaders

In [27]:
BATCH_SIZE = 64

train_dataset = MatchDataset(X_train_t, y_train_t)
val_dataset = MatchDataset(X_val_t, y_val_t)
test_dataset = MatchDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


#### Initialize the neural network

In [28]:
input_dim = X_train_t.shape[1]

model = MatchOutcomeNN(input_dim).to(device)
model


MatchOutcomeNN(
  (network): Sequential(
    (0): Linear(in_features=12, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)

#### Loss function and optimizer

In [29]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


#### Training configuration

In [30]:
EPOCHS = 20
best_val_auc = 0.0
best_state_dict = None


* 20 epochs is plenty for tabular data
* We’ll keep the best model by validation AUC

#### Training loop with validation

In [31]:
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(     # train using training data (2018-2022)
        model, 
        train_loader,
        optimizer, 
        criterion, 
        device)
    
    val_acc, val_auc = evaluate(      # evaluate on validation data (2023)
        model, 
        val_loader, 
        device)
    
    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

    # Save best model (by AUC)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state_dict = model.state_dict()
    
    

Epoch 01 | Train Loss: 0.6688 | Val Acc: 0.6077 | Val AUC: 0.6681
Epoch 02 | Train Loss: 0.6527 | Val Acc: 0.6170 | Val AUC: 0.6740
Epoch 03 | Train Loss: 0.6503 | Val Acc: 0.6191 | Val AUC: 0.6747
Epoch 04 | Train Loss: 0.6462 | Val Acc: 0.6216 | Val AUC: 0.6760
Epoch 05 | Train Loss: 0.6461 | Val Acc: 0.6217 | Val AUC: 0.6768
Epoch 06 | Train Loss: 0.6430 | Val Acc: 0.6217 | Val AUC: 0.6766
Epoch 07 | Train Loss: 0.6429 | Val Acc: 0.6226 | Val AUC: 0.6775
Epoch 08 | Train Loss: 0.6427 | Val Acc: 0.6281 | Val AUC: 0.6776
Epoch 09 | Train Loss: 0.6419 | Val Acc: 0.6276 | Val AUC: 0.6788
Epoch 10 | Train Loss: 0.6423 | Val Acc: 0.6251 | Val AUC: 0.6789
Epoch 11 | Train Loss: 0.6411 | Val Acc: 0.6274 | Val AUC: 0.6785
Epoch 12 | Train Loss: 0.6422 | Val Acc: 0.6274 | Val AUC: 0.6793
Epoch 13 | Train Loss: 0.6406 | Val Acc: 0.6288 | Val AUC: 0.6790
Epoch 14 | Train Loss: 0.6402 | Val Acc: 0.6283 | Val AUC: 0.6792
Epoch 15 | Train Loss: 0.6406 | Val Acc: 0.6276 | Val AUC: 0.6787
Epoch 16 |

* Train loss ↓ steadily
* Validation AUC ↑ steadily, no wildly fluctuations
* No sudden drops
* No divergence between train & val

#### Load best model

In [32]:
model.load_state_dict(best_state_dict)
print("Best validation AUC:", best_val_auc)


Best validation AUC: 0.6799833135117264


* +0.0084 AUC improvement over Logistic Regression (baseline) 
* Meaninful in sports prediction


The NN is likely capturing non-linear interactions such as:
 * Rank × surface
 * Cluster × height
 * Rank × cluster_diff
 * Age × tournament level

A shallow neural network was able to modestly outperform a linear baseline, suggesting the presence of non-linear relationships between player attributes and match context.

#### Evaluate on test set

In [33]:
test_acc, test_auc = evaluate(
    model,
    test_loader,
    device
)

print(f"Neural Network Test Accuracy (2024): {test_acc:.4f}")
print(f"Neural Network Test ROC-AUC (2024): {test_auc:.4f}")

Neural Network Test Accuracy (2024): 0.6299
Neural Network Test ROC-AUC (2024): 0.6782


A shallow neural network implemented in PyTorch outperformed a logistic regression baseline, improving ROC-AUC from 0.67 to 0.68 on a strictly held-out 2024 season.